In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

df_categories = spark.read.format("csv").option("header","true").load("/home/jovyan/data/categories.csv")
df_customers = spark.read.format("csv").option("header","true").load("/home/jovyan/data/customers.csv")
df_employees = spark.read.format("csv").option("header","true").load("/home/jovyan/data/employees.csv")
df_order_details = spark.read.format("csv").option("header","true").load("/home/jovyan/data/order_details.csv")
df_orders = spark.read.format("csv").option("header","true").load("/home/jovyan/data/orders.csv")
df_products = spark.read.format("csv").option("header","true").load("/home/jovyan/data/products.csv")
df_shippers = spark.read.format("csv").option("header","true").load("/home/jovyan/data/shippers.csv")
df_suppliers = spark.read.format("csv").option("header","true").load("/home/jovyan/data/suppliers.csv")

In [15]:
# 11 : Valeurs nulles

dataframes = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers,
}

for name, df in dataframes.items():
    print(f"--- {name} ---")
    for c in df.columns:
        null_count = df.filter(F.col(c).isNull()).count()
        print(f"{c} : {null_count}")

--- categories ---
category_id : 0
category_name : 0
description : 0
picture : 8
--- customers ---
customer_id : 0
company_name : 0
contact_name : 0
contact_title : 0
address : 0
city : 0
region : 60
postal_code : 1
country : 0
phone : 0
fax : 22
--- employees ---
employee_id : 0
last_name : 0
first_name : 0
title : 0
title_of_courtesy : 0
birth_date : 0
hire_date : 0
address : 0
city : 0
region : 4
postal_code : 0
country : 0
home_phone : 0
extension : 0
photo : 9
notes : 0
reports_to : 1
photo_path : 0
--- order_details ---
order_id : 0
product_id : 0
unit_price : 0
quantity : 0
discount : 0
--- orders ---
order_id : 0
customer_id : 0
employee_id : 0
order_date : 0
required_date : 0
shipped_date : 21
ship_via : 0
freight : 0
ship_name : 0
ship_address : 0
ship_city : 0
ship_region : 507
ship_postal_code : 19
ship_country : 0
--- products ---
product_id : 0
product_name : 0
supplier_id : 0
category_id : 0
quantity_per_unit : 0
unit_price : 0
units_in_stock : 0
units_on_order : 0
reord

In [16]:
# 12 : Supprimer les nulls
df_orders = df_orders.dropna(subset=["shipped_date"])

med_price = df_products.select(F.percentile_approx("unit_price", 0.5)).first()[0]
df_products = df_products.fillna({"unit_price": med_price})

In [17]:
# 13 : Cast des types

df_orders = (
    df_orders
    .withColumns({
        "order_date": F.col("order_date").cast("date"),
        "shipped_date": F.col("shipped_date").cast("date"),
        "required_date": F.col("required_date").cast("date")
    })
)

df_order_details = (
    df_order_details
    .withColumns({
        "unit_price": F.col("unit_price").cast("double"),
        "quantity": F.col("quantity").cast("int")
    })
)

In [18]:
# 14 : Nettoyage des chaînes

df_customers = (
    df_customers
    .withColumns({
        "contact_name": F.initcap(F.trim(F.col("contact_name"))),
        "country": F.upper(F.trim(F.col("country"))),
        "company_name": F.trim(F.col("company_name")),
        "contact_title": F.trim(F.col("contact_title")),
        "address": F.trim(F.col("address")),
        "city": F.trim(F.col("city")),
        "region": F.trim(F.col("region"))
    })
)

In [19]:
# 15 : Renommer les colonnes

df_order_details = (
    df_order_details
    .withColumnsRenamed({
        "unit_price": "prix_unitaire",
        "quantity": "quantite"
    })
)

df_orders = (
    df_orders
    .withColumnsRenamed({
        "ship_via": "shipper_id"
    })
)

In [20]:
# 16 : Colonnes calculées
df_order_details = (
    df_order_details
    .withColumn(
        "sous_total", 
        F.round(F.col("prix_unitaire") * F.col("quantite") * (1 - F.col("discount")),2)
    )
)

In [21]:
# 17 : Colonnes conditionnelles

df_products = (
    df_products
    .withColumn(
        "en_stock", 
        F.col("units_in_stock") > 0
    )
)

df_orders = (
    df_orders
    .withColumn(
        "is_shipped", 
        F.col("shipped_date").isNotNull()
    )
)

In [25]:
# 18 : Doublons

df_customers.select("customer_id").distinct().count()


91

In [26]:
df_customers = df_customers.dropDuplicates(subset=["customer_id"])

In [29]:
# 19 : Filtrage

df_orders = df_orders.filter(F.col("order_date").between("1997-01-01", "1997-12-31"))

df_products = df_products.filter(
    (F.col("units_in_stock") > 0) & (F.col("discontinued") == 0)
)

In [30]:
# 20 : Sélection de colonnes

df_employees = (
    df_employees
    .select(
        F.col("employee_id"),
        F.col("first_name"),
        F.col("last_name"),
        F.col("title"),
        F.col("hire_date"),
        F.col("city"),
        F.col("country")
    )
)

df_employees = (
    df_employees
    .withColumn(
        "full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name"))
    )
)

In [31]:
dataframesclean = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers,
}

PATH = "/home/jovyan/data/tmp"

for name, df in dataframesclean.items():
    output_path = f"{PATH}/{name}"
    
    df.write.mode("overwrite").parquet(output_path)
    print(f"--- {name} sauvegardé dans {output_path} ---")

--- categories sauvegardé dans /home/jovyan/data/tmp/categories ---
--- customers sauvegardé dans /home/jovyan/data/tmp/customers ---
--- employees sauvegardé dans /home/jovyan/data/tmp/employees ---
--- order_details sauvegardé dans /home/jovyan/data/tmp/order_details ---
--- orders sauvegardé dans /home/jovyan/data/tmp/orders ---
--- products sauvegardé dans /home/jovyan/data/tmp/products ---
--- shippers sauvegardé dans /home/jovyan/data/tmp/shippers ---
--- suppliers sauvegardé dans /home/jovyan/data/tmp/suppliers ---
